# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. The dataset is structured using the Croissant schema and contains detailed clinical, biochemical, imaging and genetic features for three female patients with non-classical 21-hydroxylase deficiency-associated infertility.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: {}\nDescription: {}".format(metadata.name, metadata.description))
print("Croissant Schema URL:", croissant_url)
print("Dataset Identifier (@id):", metadata.id)


## 2. Data Overview
Review available record sets, fields, and their `@id`s. The dataset consists of multiple record sets corresponding to clinical features, hormone levels + imaging, and genetic variants.

In [ ]:
# List all record sets and their @id fields
record_sets_list = list(dataset.record_sets())
print("Available record sets (@id):")
for rs in record_sets_list:
    print("- {}: {}".format(rs.id, rs.name))

# List fields in each record set
for rs in record_sets_list:
    print(f"\nFields for record set '{rs.name}' (@id: {rs.id}):")
    for f in rs.fields:
        print("  - {} (@id: {}) [dataType: {}]".format(f.name, f.id, f.data_type))

### Example: Show a few records from each record set
You can review actual data for each record set using their `@id`.

In [ ]:
# Display sample records using @id reference
for rs in record_sets_list:
    print(f"\nSample records from record set '{rs.name}' (@id: {rs.id}):")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis, using their `@id` fields.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in record_sets_list]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns from each DataFrame
for rs_id in record_set_ids:
    print(f"\nDataFrame columns for record set '@id': {rs_id}")
    print(dataframes[rs_id].columns.tolist())

# Show top rows for first record set
first_rs_id = record_set_ids[0]
print(f"\nPreview data for record set '@id': {first_rs_id}")
dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We will now perform basic data processing using field `@id`s. Let’s choose the first record set (likely clinical features) and identify a numeric field for filtering and normalization. **All fields are referenced by their `@id`**.

In [ ]:
# Choose the first record set and its numeric field (e.g., 'age_at_diagnosis')
rs = record_sets_list[0]
df = dataframes[rs.id]

# Find a numeric field in the record set
numeric_fields = [f for f in rs.fields if f.data_type in ['Integer', 'Float', 'Number']]
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    print(f"Using numeric field '@id': {numeric_field_id} ({numeric_fields[0].name})")
else:
    numeric_field_id = None

# Filter records (example threshold)
if numeric_field_id is not None:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Choose a categorical/group field (e.g., 'infertility' or similar)
    group_fields = [f for f in rs.fields if f.data_type in ['Boolean', 'Text'] and f.id != numeric_field_id]
    if group_fields:
        group_field_id = group_fields[0].id
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped (mean) data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric field found for EDA in the first record set.")

## 5. Visualization
Visualize data distributions and relationships between fields using their `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram for the numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} ({numeric_fields[0].name})")
    plt.xlabel(numeric_fields[0].name)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field was selected, visualize distribution by group
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_fields[0].name} by {group_fields[0].name}")
        plt.xlabel(group_fields[0].name)
        plt.ylabel(numeric_fields[0].name)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, processing, and visualization of the FAIR² dataset using the `mlcroissant` library.

- All data elements were referenced using their `@id` as per Croissant best practice.
- The dataset is small (N=3), with highly curated clinical/genetic variables.
- Example exploratory steps included filtering, normalization, grouping, and basic visual plots.

This dataset is ideal as an academic case example for genotype–phenotype analysis in rare endocrine disorders, but is not suitable for direct predictive AI model training due to cohort constraints.